In [4]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\Robin Yadav\oulad-early-warning")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
CUTOFF = 28  # end of week 4, in days from course start

print("raw folder found:", RAW.exists())

info = pd.read_csv(RAW / "studentInfo.csv")
reg = pd.read_csv(RAW / "studentRegistration.csv")

key = ["code_module", "code_presentation", "id_student"]
print("duplicate registrations:", info.duplicated(key).sum())
print("unique students:", info["id_student"].nunique(), "of", len(info), "rows")
print(reg.isna().sum())

raw folder found: True
duplicate registrations: 0
unique students: 28785 of 32593 rows
code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64


In [5]:
df = info.merge(reg, on=key, how="left", validate="one_to_one")
df["at_risk"] = df["final_result"].isin(["Withdrawn", "Fail"]).astype(int)

print("Target balance before cutoff:")
print(df["at_risk"].value_counts(normalize=True).round(3))

left_early = df["date_unregistration"].notna() & (df["date_unregistration"] <= CUTOFF)
print("\nWithdrew on or before day", CUTOFF, ":", left_early.sum())

df = df[~left_early].copy()
print("Rows kept:", len(df))
print("\nTarget balance after cutoff:")
print(df["at_risk"].value_counts(normalize=True).round(3))

Target balance before cutoff:
at_risk
1    0.528
0    0.472
Name: proportion, dtype: float64

Withdrew on or before day 28 : 5055
Rows kept: 27538

Target balance after cutoff:
at_risk
0    0.559
1    0.441
Name: proportion, dtype: float64


In [6]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED / f"base_day{CUTOFF}.csv", index=False)
print("saved", df.shape)

saved (27538, 15)


In [7]:
import os
print(os.getcwd())

C:\Users\Robin Yadav
